# 1장 1강: HTML과 JSON 구조 — 실습문제

## 실습 목표

- HTML에서 태그, 속성, 요소의 내용과 부모·자식 관계를 구분할 수 있다.
- 책 하나를 묶는 반복 영역을 확인하고 제목·가격·링크의 위치를 설명할 수 있다.
- JSON의 Object와 Array를 구분하고 Python의 `dict`, `list`와 연결할 수 있다.
- `json.loads()`와 Key·인덱스를 이용해 기본 구조와 중첩 구조에서 값을 확인할 수 있다.
- HTML과 JSON의 표현 방식을 비교하고 레코드·필드·출처 정보를 구분할 수 있다.

## 실습 환경 / 데이터

- Python 3, Jupyter Notebook 또는 Google Colab, 웹브라우저
- Python 기본 모듈 `json` 사용: 별도 패키지 설치나 사전 CSV 파일이 필요하지 않습니다.
- HTML 관찰: [Books to Scrape](https://books.toscrape.com/)
- JSON 관찰: [JSONPlaceholder 게시글 목록](https://jsonplaceholder.typicode.com/posts)
- HTML은 Books to Scrape 첫 페이지의 첫 두 책에서 필요한 부분만 남긴 예시입니다. 이미지·평점·재고·버튼 등을 생략했으며, 제목·가격·링크와 해당 요소의 계층 관계는 유지했습니다.
- `posts_json_text`는 2026-11-02에 확인한 `/posts` 응답의 첫 두 게시글입니다. 두 건은 전체 데이터가 아니라 학습용 발췌본입니다.
- `nested_json_text`, `book_json_text`는 중첩 구조와 표현 비교를 위해 **직접 재구성한 학습용 JSON**입니다. 사이트가 이 형태의 JSON을 반환한다는 뜻이 아닙니다.

| 데이터 | 한 항목의 의미 | 주요 관찰 대상 |
|---|---|---|
| 책 HTML | 책 한 권 | `article`, `h3`, `a`, `p`와 `class`, `title`, `href` |
| 게시글 JSON | 게시글 한 개 | `userId`, `id`, `title`, `body` |
| 재구성 JSON | 게시글 또는 책 한 개 | 바깥 Object, 내부 Array, 중첩 Object |

> 이번 강의는 **구조 확인과 간단한 값 접근** 단계입니다. HTML 자동 추출, 반복 수집, DataFrame 변환, CSV·DB 저장은 이번 문제의 요구사항이 아닙니다. 강의의 명언 예시를 책과 게시글 사례로 바꾸어 같은 개념을 연습합니다.


## 시작 전에 읽을 기초 개념

이번 강의에서는 **수집할 값이 어디에 있고, 어떤 구조로 묶여 있는지** 확인합니다. 책 제목 하나를 찾더라도 제목이 화면의 글자인지, HTML 속성 안에 있는지, JSON 객체 안에 있는지에 따라 읽는 방법이 다릅니다.

| 용어 | 쉽게 풀어쓴 뜻 | 이번 실습에서의 예 |
|---|---|---|
| HTML | 웹페이지의 요소와 포함 관계를 표시하는 문서 형식 | 책 목록을 구성하는 article·h3·a·p |
| 태그 / 요소 | 태그는 요소를 표시하는 이름·표기이며, 요소는 시작 태그·내용·종료 태그 등을 포함한 구성 단위 | `<p class="price_color">£51.77</p>`라는 가격 요소 |
| 속성 / 속성값 | 요소에 붙인 추가 정보의 이름 / 실제 설정한 값 | class / product_pod, href / 링크 경로 |
| 부모·자식 / 조상·자손 | 바로 한 단계 바깥·안쪽 / 여러 단계의 포함 관계 | 가격 p의 부모는 div, 더 바깥 조상은 article |
| DOM | 브라우저가 HTML 요소들의 관계를 구조로 표현한 것 | Elements에서 제목 요소와 바깥 영역을 확인 |
| JSON | 값을 구조화해서 표현하고 전달하는 텍스트 형식 | 게시글 번호·제목·본문을 묶은 응답 |
| Key / Value | 항목의 이름 / 그 항목에 들어 있는 값 | `"id": 2`에서 id가 Key, 2가 Value |
| Object / dict | 이름과 값을 쌍으로 묶은 구조 / 파싱 후의 Python 딕셔너리 | 게시글 하나의 id·userId·title·body |
| Array / list | 여러 값을 순서대로 담은 구조 / 파싱 후의 Python 리스트 | 게시글 두 개가 들어 있는 목록 |
| 인덱스 | 목록에서 항목이 놓인 위치 번호 | 0은 첫 번째, 1은 두 번째 |
| 파싱 | 텍스트의 규칙을 읽어 프로그램에서 다룰 구조로 바꾸는 것 | json.loads()로 JSON 문자열을 list·dict로 변환 |
| 중첩 | 값의 묶음 안에 또 다른 묶음이 들어 있는 상태 | posts 목록의 게시글 안에 author 객체가 있음 |
| 레코드 / 필드 | 대상 하나의 정보 묶음 / 그 안의 개별 항목 | 책 한 권 / 제목·가격·링크 |
| 스키마 | 사용할 필드의 이름·의미·자료형 등을 정한 구조 | title·price_text·detail_href·source_url 설계표 |

**자료형**은 값의 종류입니다. 이번에는 글자 `str`, 정수 `int`, 참·거짓 `bool`, 목록 `list`, 이름과 값의 묶음 `dict`를 확인합니다. 값이 없음을 나타내는 None의 자료형은 NoneType입니다. type()은 종류를 확인하며, 값을 원하는 종류로 바꾸는 함수는 아닙니다.

### 같은 괄호도 무엇을 표현하는지 구분하세요

| 위치 | 표현 | 뜻 |
|---|---|---|
| JSON 텍스트 안 | `{ "id": 2 }` | Key와 Value를 묶은 Object |
| JSON 텍스트 안 | `[ {...}, {...} ]` | 두 항목이 순서대로 들어 있는 Array |
| 파싱한 Python 값에 접근 | `posts[1]` | list에서 두 번째 항목 선택 |
| 파싱한 Python 값에 접근 | `first_post["title"]` | dict에서 title Key에 해당하는 값 선택 |

**중요:** 따옴표 안에 JSON 모양의 글자가 있다고 곧바로 Python 목록이나 딕셔너리가 되는 것은 아닙니다. 준비 셀에서는 문자열로 보관하고, 뒤에서 json.loads()로 해석한 결과에 Key·인덱스를 사용합니다.

### 전체 실습 흐름

| 단계 | 입력 | 하는 일 | 확인할 결과 |
|---|---|---|---|
| 준비 | 제공된 HTML·JSON 예시 | 변수에 문자열로 보관 | 아직 파싱 전인 예시 자료 |
| 필수 1 | book_html | 책 영역·태그·속성·포함 관계를 관찰 | 같은 책의 정보가 어디에 있는지 |
| 필수 2 | 게시글 JSON 문자열 두 종류 | 파싱 후 Key·인덱스로 값 확인 | 전체 목록과 한 게시글, 중첩 객체의 차이 |
| 과제 1 | 같은 책을 표현한 HTML·JSON | 두 번째 책의 값을 비교하고 필드 설계 | 한 행의 대상과 값의 의미·형식·출처 |

이번 단계는 **구조 읽기와 값 확인**입니다. 자동으로 사이트를 반복 수집하거나 CSV·DB에 저장하는 코드를 추가하지 않습니다.


## 실습 준비

1. 위 두 사이트를 브라우저에서 엽니다. 책 목록과 게시글 JSON이 각각 어떻게 보이는지 확인하세요.
2. Books to Scrape에서 첫 번째 책 제목을 우클릭하여 **검사**를 선택하거나, `F12` → `Elements` → 요소 선택 도구를 사용하세요. 제목을 감싼 요소에서 바깥쪽 책 영역까지 살펴보세요.
3. 아래 제공 코드를 위에서부터 실행하여 `json` 모듈과 HTML·JSON 예시 문자열을 준비하세요. 이 셀은 수정할 필요가 없습니다.
4. HTML에서는 데이터가 들어 있는 태그·속성을, JSON에서는 바깥 괄호와 각 항목의 Key를 먼저 확인하세요.

**진행 안내**

- 제공 코드에는 필요한 예시가 모두 포함되어 있어 노트북 실행에 인터넷 연결은 필요하지 않습니다. 사이트에 접속할 수 없는 경우 제공된 예시를 기준으로 답하고, 브라우저 확인을 하지 못했다고 적으세요.
- 실시간 사이트의 내용이 바뀌면 관찰 내용과 예시의 차이를 기록하세요. 이 노트북의 출력 예시와 정답은 내장된 자료 기준입니다.
- `print()`로 보는 HTML은 태그가 포함된 문자열입니다. 브라우저처럼 웹페이지를 그려 주는 기능이 아닙니다.
- 이 단계에서는 결측치를 제거하거나 가격을 숫자로 바꾸지 않습니다. 원본 값이 어디에 있고 어떤 형태인지 확인합니다.


### 코드를 읽는 순서와 기호

준비 셀 → 필수 1 → 필수 2 → 과제 1 순서로 실행하세요. 셀은 한 번에 실행하는 코드 상자이며, 앞 셀에서 만든 변수는 뒤 셀에서 사용합니다. 준비 셀을 실행하지 않으면 book_html이나 posts_json_text라는 이름을 찾지 못할 수 있습니다.

| 코드 표현 | 말로 읽기 | 이번 실습에서의 주의점 |
|---|---|---|
| `# 설명` | 사람이 읽는 주석 | Python이 실행하는 계산이 아님 |
| `이름 = 결과` | 오른쪽 결과를 왼쪽 이름으로 보관 | 수학의 등식 증명과 다름 |
| `"title"` | title이라는 글자 | 따옴표 없는 title 변수와 다름 |
| `print(값)` | 값을 화면에 표시 | 화면 표시가 파일 저장을 뜻하지 않음 |
| `type(값)` | 값의 자료형 확인 | `<class 'list'>`는 list라는 종류라는 뜻 |
| `len(값)` | 대상에 들어 있는 항목 수 확인 | list는 항목 수, dict는 Key 수를 셈 |
| `json.loads(문자열)` | JSON 문자열을 해석하여 Python 값으로 변환 | 웹주소를 요청하거나 파일을 읽는 코드가 아님 |
| `목록[0]`, `목록[1]` | 첫 번째·두 번째 항목 선택 | 게시글 id를 검색하는 것이 아니라 위치를 선택 |
| `객체["Key"]` | 그 이름에 해당하는 값 선택 | Key의 철자와 대소문자가 실제 구조와 같아야 함 |
| `객체.keys()` | 현재 객체의 항목 이름 확인 | 모든 중첩 단계의 Key를 한꺼번에 모으지 않음 |
| `list(객체.keys())` | 항목 이름들을 출력하기 쉬운 목록으로 변환 | 게시글 전체를 새로 수집하는 동작이 아님 |

**출력 이름표와 실제 값을 구분하세요.** `print("전체 자료형:", type(posts))`에서 따옴표 안은 사람이 읽는 이름표입니다. 실제 자료형 정보는 쉼표 뒤의 type(posts)가 계산합니다. 이름표를 바꾸어도 posts의 내용은 바뀌지 않습니다.


In [2]:
# JSON 텍스트를 Python 값으로 해석할 기본 모듈을 불러옵니다.
import json

# 출처 주소를 문자열로 보관합니다. 이 대입만으로 웹 요청이 일어나지는 않습니다.
BOOKS_URL = 'https://books.toscrape.com/'
POSTS_URL = 'https://jsonplaceholder.typicode.com/posts'

# 실제 책 HTML에서 학습에 필요한 요소만 남긴 문자열입니다.
book_html = r"""
<article class="product_pod">
  <h3><a href="catalogue/a-light-in-the-attic_1000/index.html" title="A Light in the Attic">A Light in the ...</a></h3>
  <div class="product_price">
    <p class="price_color">£51.77</p>
  </div>
</article>

<article class="product_pod">
  <h3><a href="catalogue/tipping-the-velvet_999/index.html" title="Tipping the Velvet">Tipping the Velvet</a></h3>
  <div class="product_price">
    <p class="price_color">£53.74</p>
  </div>
</article>
"""

# /posts 응답의 첫 두 게시글을 발췌한 JSON 문자열입니다.
posts_json_text = r"""
[
  {
    "userId": 1,
    "id": 1,
    "title": "sunt aut facere repellat provident occaecati excepturi optio reprehenderit",
    "body": "quia et suscipit\nsuscipit recusandae consequuntur expedita et cum\nreprehenderit molestiae ut ut quas totam\nnostrum rerum est autem sunt rem eveniet architecto"
  },
  {
    "userId": 1,
    "id": 2,
    "title": "qui est esse",
    "body": "est rerum tempore vitae\nsequi sint nihil reprehenderit dolor beatae ea dolores neque\nfugiat blanditiis voluptate porro vel nihil molestiae ut reiciendis\nqui aperiam non debitis possimus qui neque nisi nulla"
  }
]
"""

# 실제 API 원본이 아닌 학습용 재구성입니다.
# userId 값을 author 안의 user_id로 옮기고 출처·예시 표시를 추가했습니다.
nested_json_text = r"""
{
  "source_url": "https://jsonplaceholder.typicode.com/posts",
  "posts": [
    {
      "id": 1,
      "title": "sunt aut facere repellat provident occaecati excepturi optio reprehenderit",
      "author": {
        "user_id": 1
      }
    },
    {
      "id": 2,
      "title": "qui est esse",
      "author": {
        "user_id": 1
      }
    }
  ],
  "is_example": true,
  "note": null
}
"""

# HTML과 같은 책 정보를 중첩 JSON으로 직접 표현한 학습용 예시입니다.
book_json_text = r"""
{
  "source_url": "https://books.toscrape.com/",
  "books": [
    {
      "title": "A Light in the Attic",
      "price": {
        "text": "£51.77"
      },
      "detail_href": "catalogue/a-light-in-the-attic_1000/index.html"
    },
    {
      "title": "Tipping the Velvet",
      "price": {
        "text": "£53.74"
      },
      "detail_href": "catalogue/tipping-the-velvet_999/index.html"
    }
  ]
}
"""

print("실습 준비 완료: HTML 예시 1개, JSON 예시 3개")
print("책 목록 출처:", BOOKS_URL)
print("게시글 출처:", POSTS_URL)


실습 준비 완료: HTML 예시 1개, JSON 예시 3개
책 목록 출처: https://books.toscrape.com/
게시글 출처: https://jsonplaceholder.typicode.com/posts


### 준비 코드 상세 해설 — 무엇을 보관하고 있나요?

준비 셀은 **실습에 사용할 예시를 메모리에 준비하는 코드**입니다. 주소가 적혀 있어도 네트워크 요청을 보내는 줄은 없습니다.

| 코드·변수 | 하는 일 | 준비 직후의 상태 |
|---|---|---|
| `import json` | Python 기본 JSON 도구를 불러옴 | 이후 json.loads() 사용 가능 |
| `BOOKS_URL`, `POSTS_URL` | 자료를 확인한 출처 주소 보관 | 둘 다 str; 주소를 보관한 것만으로 접속하지 않음 |
| `book_html` | 두 책의 HTML을 여러 줄 문자열로 보관 | 태그와 속성이 포함된 str |
| `posts_json_text` | 원본 게시글 응답의 두 항목을 문자열로 보관 | 파싱 전 str; 바깥 모양은 Array |
| `nested_json_text` | 중첩 구조 설명용으로 재구성한 문자열 | 파싱 전 str; 바깥 모양은 Object |
| `book_json_text` | HTML과 같은 책 정보를 JSON으로 재구성한 문자열 | 파싱 전 str; 실제 책 API 응답이 아님 |
| 마지막 세 print() | 준비 완료 문구와 출처 주소 출력 | 데이터가 저장되었다거나 수집 완료됐다는 뜻이 아님 |

#### 여러 줄 문자열과 r 표기 읽기

준비 셀의 `r` 뒤에는 큰따옴표 세 개가 이어집니다. 큰따옴표 세 개는 여러 줄의 글자를 한 문자열로 담는 표기입니다. 앞의 r은 역슬래시 표기를 Python 문자열 단계에서 그대로 두기 위해 사용합니다. 준비 코드는 제공된 그대로 실행하면 되며, 이 표기를 새로 작성하는 문제는 아닙니다.

게시글 body의 줄바꿈을 예로 들면 다음 순서입니다.

1. 준비 셀의 JSON 텍스트에는 역슬래시와 n으로 적힌 `\n`이 들어 있습니다.
2. json.loads()가 JSON을 해석하면서 이 표시를 본문 문자열 안의 줄바꿈으로 바꿉니다.
3. 그 body를 print()하면 본문이 여러 줄에 보입니다.

줄이 여러 개로 출력되어도 body는 여전히 **게시글 하나의 본문 문자열**입니다. 줄바꿈 개수를 게시글 개수로 세지 않습니다.

#### 예시의 출처와 역할 구분

posts_json_text는 실제 응답의 일부를 발췌한 것이고, nested_json_text·book_json_text는 구조를 설명하려고 다시 구성한 예시입니다. 재구성 데이터에 author나 source_url을 추가했다고 원본 API에도 그 항목이 있다고 판단하면 안 됩니다.


---

## 필수 1. HTML의 태그·속성과 반복 구조 확인

### 문제 1-1. 책 한 권의 정보는 HTML의 어디에 있을까요?

#### 문제 설명

책 제목·가격·상세 페이지 링크를 수집하려고 합니다. 자동 추출 코드를 작성하기 전에, Books to Scrape에서 책 한 권을 묶는 영역과 각 정보의 위치를 먼저 확인하세요. 아래 제공 HTML과 브라우저의 Elements 화면을 비교합니다.

#### 요구사항

1. 아래 제공 코드를 실행하여 책 두 권의 HTML을 확인하세요. 브라우저에서도 같은 요소를 찾아보세요.
2. 관찰표에 책 한 권의 영역, 제목, 가격, 상세 페이지 링크의 **태그·속성·내용 또는 속성값**을 기록하세요.
3. 첫 번째 책의 전체 제목, 표시된 가격 문자열, 제목 링크의 `href` 값을 기록하세요. 가격과 주소는 원본 문자열 그대로 적으세요.
4. `article` → `h3` → `a`, `article` → `div` → `p`의 포함 관계를 설명하세요. 직접 부모와 더 바깥 조상을 구분하세요.
5. 제공 HTML에서 책 단위 영역이 몇 번 반복되는지 기록하고, 이 값이 실제 사이트 전체 책 수인지 설명하세요.
6. Q1~Q4에 답하세요.

> `article`은 이 사이트에서 책 한 권을 묶는 데 사용한 태그입니다. `article.product_pod`는 `article` 태그 중 `product_pod` 클래스를 가진 요소를 가리키는 표기입니다. 이번에는 선택자로 자동 추출할 필요가 없습니다.

#### 해석 질문

**Q1.** 태그와 속성은 어떻게 다른가요? `a`, `title`, `href`를 예로 들어 설명하세요.

**Q2.** 첫 번째 책의 전체 제목을 확인할 때, 화면에 보이는 텍스트와 `title` 속성 중 무엇을 확인해야 하나요? 이유도 설명하세요.

**Q3.** 제시된 HTML에서 `h3`와 가격을 담은 `p`는 `article`의 직접 자식인가요? 직접 부모를 각각 적으세요.

**Q4.** 여러 책을 구분할 때 책 한 권을 묶는 영역을 먼저 찾는 이유는 무엇인가요?

#### 제출 결과

- HTML 확인 코드 실행 결과
- 관찰표와 첫 번째 책의 제목·가격·상세 링크
- 계층 관계와 반복 영역 개수 설명
- 브라우저 확인 여부와 Q1~Q4 답변


### 작성 도움말

책 한 권의 영역부터 찾은 뒤, 같은 영역 안에서 제목과 가격을 확인하세요. 관찰표에는 **어느 요소인지 → 속성인지 내용인지 → 실제 값이 무엇인지** 순서로 기록하면 됩니다. 제목 a와 가격 p의 바로 바깥 요소를 각각 확인하면 직접 부모를 구분할 수 있습니다.

print(book_html)은 이미 완성된 제공 코드입니다. 복잡한 추출 코드를 새로 만들 필요 없이 출력과 Elements 화면을 읽고 답하세요.


### 제공 코드: HTML 확인

아래 코드는 완성되어 있습니다. 실행 후 관찰 및 답변 작성란을 채우세요.


In [3]:
# 준비된 HTML 문자열을 그대로 보여 줍니다. 자동 추출이나 새 요청은 아닙니다.
print(book_html)



<article class="product_pod">
  <h3><a href="catalogue/a-light-in-the-attic_1000/index.html" title="A Light in the Attic">A Light in the ...</a></h3>
  <div class="product_price">
    <p class="price_color">£51.77</p>
  </div>
</article>

<article class="product_pod">
  <h3><a href="catalogue/tipping-the-velvet_999/index.html" title="Tipping the Velvet">Tipping the Velvet</a></h3>
  <div class="product_price">
    <p class="price_color">£53.74</p>
  </div>
</article>



### 필수 1 관찰 및 답변 작성란

브라우저 확인 여부(확인 / 접속 불가): 확인  

| 확인 대상 | 태그와 속성 | 내용 또는 속성값 |
|---|---|---|
| 책 한 권의 영역 | `article` = `class="product_pod"` | 한 영역에 책 한 권의 관련 요소 포함 |
| 첫 번째 책의 전체 제목 | `h3`안에 `a` 의 `title` 속성 | A Light in the Attic |
| 첫 번째 책의 가격 | `p`,`class="price_color"`의 텍스트 부분 | £51.77 |
| 첫 번째 책의 상세 링크 | 제목을 감싼 `a`의 `href` 속성 | catalogue/a-light-in-the-attic_1000/index.html |

- 제목 요소의 포함 관계: `article.product_pod` → `h3` → `a` (a의 직접 부모는 h3, article은 더 바깥 조상)  
- 가격 요소의 포함 관계: `article.product_pod` → `div.product_price` → `p.price_color` (p의 직접 부모는 div, article은 더 바깥 조상)  
- 제공 HTML의 책 영역 개수: 2개 (
    article.product_pod가 2번 반복)  
- 실제 사이트 전체 건수와의 차이: 실제 페이지에는 더 다양한 태그 a,p,ui,li등이 있음  
-> 따라서 정확하게 무엇을 추출하고 싶은지 파악하고 해당 태그의 위치를 명확히 파악하는 것이 중요

**Q1. 태그와 속성은 어떻게 다른가요? `a`, `title`, `href`를 예로 들어 설명하세요.**

- **태그** : 요소의 종류를 나타내는 태그. `a`는 링크 요소라는 뜻임  
- **속성** : 시작 태그 안에 `이름="값"` 형태로 붙는 추가 정보를 붙이는 속성의 이름  
        -> 예제에서는 제목 링크의 title 속성의 책에 전체 제목을 넣어두었음  
    -> 태그는 "무슨 요소인지", 속성은 "그 요소에 대한 추가 정보"를 나타냄  

**Q2. 첫 번째 책의 전체 제목을 확인할 때, 화면에 보이는 텍스트와 `title` 속성 중 무엇을 확인해야 하나요? 이유도 설명하세요.**

- **첫번째 책은 `title` 속성을 확인해야 함**  
    요소 안의 표시 텍스트는 `A Light in the ...`로 보이지만 title 속성에는 `A Light in the Attic`이 들어있다.  
    -> 모든 사이트가 전체 제목을 title로 쓰는 것은 아니기 때문에 실제 구조를 확인해야한다.  

**Q3. 제시된 HTML에서 `h3`와 가격을 담은 `p`는 `article`의 직접 자식인가요? 직접 부모를 각각 적으세요.**

 
- **`h3` : article의 직접 자식임** (직접 부모 : `article.product_pod`)  
- **가격 `p` : article의 직접 자식이 아님** (직접 부모 : `div.product_price`)  
        -> article의 가격p의 바깥 조상이며 직접 부모는 아니다.  
        -> 직접 자식은 바로 한 단계 안쪽 요소를 뜻하고 자손은 여러 단계 안쪽 요소까지 포함한다. 

**Q4. 여러 책을 구분할 때 책 한 권을 묶는 영역을 먼저 찾는 이유는 무엇인가요?**

- 제목과 가격이 같은 책에 속하는지 확인하기 위해서  
-> 책마다 article.product_pod 영역이 반복되므로 한 영역 안에서 제목과 가격을 확인하면  
-> 서로 다른 책의 정보를 섞을 가능성을 줄일 수 있다.

---

## 필수 2. JSON 구조와 중첩된 값 확인

### 문제 2-1. 게시글 JSON의 구조와 필요한 값의 위치 확인

#### 문제 설명

JSONPlaceholder의 `/posts`는 여러 게시글을 배열로 제공합니다. 준비 셀의 `posts_json_text`에는 실제 응답의 첫 두 게시글이 들어 있습니다. 원본 구조를 확인한 뒤, 학습용으로 재구성한 `nested_json_text`에서 중첩된 값에 접근하세요.

#### 요구사항

1. `posts_json_text`를 `json.loads()`로 파싱하여 `posts`에 저장하세요.
2. `posts`의 자료형과 게시글 개수, 첫 번째 게시글의 자료형과 Key 목록을 출력하세요.
3. 첫 번째 게시글의 `id`, `userId`, `title`, `body`를 각각 출력하세요.
4. 두 번째 게시글의 제목을 인덱스와 Key를 이용하여 출력하세요. 첫 번째 게시글의 `id`와 `title` 값의 Python 자료형도 출력하세요.
5. `nested_json_text`를 파싱하여 `nested_data`에 저장하고, 전체 자료형·최상위 Key 목록·`posts` 목록의 개수를 출력하세요.
6. 재구성 예시에서 두 번째 게시글의 `author` 내부 `user_id` 값을 출력하세요. `is_example`과 `note`의 값과 Python 자료형도 출력하세요.
7. `id`, `userId`, `title`, `body`의 의미와 Python 자료형을 표로 정리하고, Q1~Q5에 답하세요.

> 원본 `/posts`는 배열로 시작하며 `posts`라는 최상위 Key가 없습니다. 학습용 재구성 예시는 객체로 시작하며 `posts` Key가 있습니다. 두 구조를 구분하세요. `author`는 원본 응답의 필드가 아니며, 이 예시에서는 작성자 **이름**이 아닌 작성자 **번호**를 담습니다.

#### 해석 질문

**Q1.** 원본 `/posts`의 가장 바깥 구조와 `nested_json_text`의 가장 바깥 구조는 어떻게 다른가요?

**Q2.** `posts[1]`과 `posts["title"]` 중 올바른 접근은 무엇인가요? 두 번째 게시글의 제목에 도달하는 순서로 설명하세요.

**Q3.** `id`와 `userId`는 각각 무엇을 식별하나요? 두 게시글의 `userId`가 같으면 같은 게시글인가요?

**Q4.** 원본 게시글에 `author.name`이 없는데 작성자 이름을 추정해서 적어도 되나요? 재구성 예시의 `author.user_id`는 무엇을 나타내나요?

**Q5.** `true`, `null`은 파싱 후 어떻게 바뀌나요? `source_url`과 `is_example`은 원본 `/posts`의 필드인가요?

#### 제출 결과

- 두 JSON의 파싱 코드와 요구한 출력 결과
- 원본 게시글의 필드 의미·자료형 표
- Q1~Q5 답변


### 작성 도움말

1. 문자열을 먼저 json.loads()로 해석하세요.
2. type()으로 전체가 list인지 dict인지 확인하세요.
3. list이면 위치 번호로 항목을 고르고, 선택한 dict에서는 Key로 값을 찾으세요.
4. 재구성 예시는 바깥에 객체가 하나 더 있으므로 게시글 목록이 어느 Key 아래 있는지 먼저 확인하세요.

`type(...)`은 종류, `len(...)`은 항목 수, `.keys()`는 항목 이름을 확인합니다. 각각 무엇을 확인하는지 출력 이름표에 구분해 적으세요. userId와 user_id는 예시 구조에 따라 철자가 다르므로 원본을 보고 선택하세요.


### 제공 코드: 비교할 JSON 확인


In [4]:
print("[원본 응답에서 발췌한 JSON]")
print(posts_json_text)
print("\n[학습용으로 재구성한 JSON]")
print(nested_json_text)


[원본 응답에서 발췌한 JSON]

[
  {
    "userId": 1,
    "id": 1,
    "title": "sunt aut facere repellat provident occaecati excepturi optio reprehenderit",
    "body": "quia et suscipit\nsuscipit recusandae consequuntur expedita et cum\nreprehenderit molestiae ut ut quas totam\nnostrum rerum est autem sunt rem eveniet architecto"
  },
  {
    "userId": 1,
    "id": 2,
    "title": "qui est esse",
    "body": "est rerum tempore vitae\nsequi sint nihil reprehenderit dolor beatae ea dolores neque\nfugiat blanditiis voluptate porro vel nihil molestiae ut reiciendis\nqui aperiam non debitis possimus qui neque nisi nulla"
  }
]


[학습용으로 재구성한 JSON]

{
  "source_url": "https://jsonplaceholder.typicode.com/posts",
  "posts": [
    {
      "id": 1,
      "title": "sunt aut facere repellat provident occaecati excepturi optio reprehenderit",
      "author": {
        "user_id": 1
      }
    },
    {
      "id": 2,
      "title": "qui est esse",
      "author": {
        "user_id": 1
      }
    }
  ],
  "

In [5]:
# 필수 2 코드를 작성하세요.
# 준비 셀의 posts_json_text와 nested_json_text를 사용하세요.

# 1. 원본 JSON 파싱 → 전체 구조·첫 항목 구조 확인
# 1. `posts_json_text`를 `json.loads()`로 파싱하여 `posts`에 저장하세요.
posts = json.loads(posts_json_text)

# 2. `posts`의 자료형과 게시글 개수, 첫 번째 게시글의 자료형과 Key 목록을 출력하세요.
first_post = posts[0]
print("전체 자료형:", type(posts))
print("게시글 개수:", len(posts))
print("첫 번째 게시글 자료형:", type(first_post))
print("첫 번째 게시글 Key 목록:", list(first_post.keys()))

# 2. 첫 게시글의 필드와 두 번째 제목·자료형 확인
# 3. 첫 번째 게시글의 `id`, `userId`, `title`, `body`를 각각 출력하세요.
print("\n[첫 번째 게시글]")
print("id:", first_post["id"])
print("userId:", first_post["userId"])
print("title:", first_post["title"])
print("body:")
print(first_post["body"])

# 4. 두 번째 게시글의 제목을 인덱스와 Key를 이용하여 출력하세요. 첫 번째 게시글의 `id`와 `title` 값의 Python 자료형도 출력하세요.
print("\n두 번째 게시글 제목:", posts[1]["title"])
print("첫 번째 게시글 id 자료형:", type(first_post["id"]))
print("첫 번째 게시글 title 자료형:", type(first_post["title"]))

# 3. 재구성 JSON 파싱 → 목록·중첩 값·논리 값·값 없음 확인
# 5. `nested_json_text`를 파싱하여 `nested_data`에 저장하고, 전체 자료형·최상위 Key 목록·`posts` 목록의 개수를 출력하세요.
nested_data = json.loads(nested_json_text)
print("\n재구성 전체 자료형:", type(nested_data))
print("최상위 Key 목록:", list(nested_data.keys()))
print("posts 목록 개수:", len(nested_data["posts"]))

# 6. 재구성 예시에서 두 번째 게시글의 `author` 내부 `user_id` 값을 출력하세요. `is_example`과 `note`의 값과 Python 자료형도 출력하세요.
print("두 번째 게시글 author.user_id:", nested_data["posts"][1]["author"]["user_id"])
print("is_example 값:", nested_data["is_example"], "/ 자료형:", type(nested_data["is_example"]))
print("note 값:", nested_data["note"], "/ 자료형:", type(nested_data["note"]))

전체 자료형: <class 'list'>
게시글 개수: 2
첫 번째 게시글 자료형: <class 'dict'>
첫 번째 게시글 Key 목록: ['userId', 'id', 'title', 'body']

[첫 번째 게시글]
id: 1
userId: 1
title: sunt aut facere repellat provident occaecati excepturi optio reprehenderit
body:
quia et suscipit
suscipit recusandae consequuntur expedita et cum
reprehenderit molestiae ut ut quas totam
nostrum rerum est autem sunt rem eveniet architecto

두 번째 게시글 제목: qui est esse
첫 번째 게시글 id 자료형: <class 'int'>
첫 번째 게시글 title 자료형: <class 'str'>

재구성 전체 자료형: <class 'dict'>
최상위 Key 목록: ['source_url', 'posts', 'is_example', 'note']
posts 목록 개수: 2
두 번째 게시글 author.user_id: 1
is_example 값: True / 자료형: <class 'bool'>
note 값: None / 자료형: <class 'NoneType'>


### 필수 2 답변 작성란

**원본 게시글의 필드 의미·자료형**

| 필드 | 의미 | Python 자료형 |
|---|---|---|
| `id` | 게시글 식별 번호 | int |
| `userId` | 게시글 작성자 식별 번호 | int |
| `title` | 게시글 제목 | str |
| `body` | 게시글 본문 (줄바꿈 포함) | str |

**Q1. 원본 `/posts`의 가장 바깥 구조와 `nested_json_text`의 가장 바깥 구조는 어떻게 다른가요?**

- 답변:  
    - **원본 `/posts`** :  `[ ]`로 시작하는 **Array**이므로 파싱하면 **list** 형태  
    - **`nested_json_text`** : 가장 바깥이 `{ }`로 시작하는 **Object**이고, 파싱하면 **dict** 형태 
    -> 재구성 예시에서는 먼저 nset_data["posts"]로 내부 목록을 꺼내야 한다. 

**Q2. `posts[1]`과 `posts["title"]` 중 올바른 접근은 무엇인가요? 두 번째 게시글의 제목에 도달하는 순서로 설명하세요.**

- 답변: **`posts[1]`이 올바른 접근임**  
    `posts`는 list이르모 posts[1]이 올바른 접근, 인덱스는 0부터 시작하니까 1은 두번째 게시글을 선택한다.
    -> 선택한 게시글은 딕셔너리이므로 이어서 ["title"]을 찾아 제목을 완성한다.  
    -> 따라서 posts[1]["title"]을 가져오면 된다.

**Q3. `id`와 `userId`는 각각 무엇을 식별하나요? 두 게시글의 `userId`가 같으면 같은 게시글인가요?**

- 답변:  
    - **`id`** : 게시글 번호  
    - **`userId`** : 게시글을 쓴 작성자 번호  
    -> u예시에서 두 게시글의 작성자 번호는 모두 1이지만 게시글 번호는 1과 2로 다르다.  
    -> 즉, 한 사람이 여러 게시글을 작성했다는 뜻  

**Q4. 원본 게시글에 `author.name`이 없는데 작성자 이름을 추정해서 적어도 되나요? 재구성 예시의 `author.user_id`는 무엇을 나타내나요?**

- 답변: **추정해서 적으면 안 됨**  
    원본 게시글에는 작성자 번호(userId)만 있기 때문에 이 자료만으로 이름을 알 수 없다.  
    -> 재구성 예시의 `author.user_id`는 원본의 `userId` 값을 중첩된 위치에 옮겨둔 것  

**Q5. `true`, `null`은 파싱 후 어떻게 바뀌나요? `source_url`과 `is_example`은 원본 `/posts`의 필드인가요?**

- 답변:  
    - **`true`** → Python의 **`True`** (자료형 bool)  
    - **`null`** → Python의 **`None`** (자료형 NoneType)  
    - `source_url`, `is_example`은 원본 게시글에 없는 학습용 추가 정보


---

## 과제 1. HTML·JSON 비교와 데이터 필드 설계

### 문제 3-1. 두 번째 책의 HTML·JSON 비교와 필드 설계

#### 문제 설명

필수 문제에서 연습한 방법으로 **두 번째 책**의 제목·가격·상세 링크를 확인하세요. `book_json_text`는 필수 1의 두 책을 JSON으로 직접 표현한 학습용 자료입니다. Books to Scrape에서 제공하는 API 응답이 아닙니다.

필수 1의 HTML 관찰과 필수 2의 Key·인덱스 접근을 그대로 활용하면 해결할 수 있습니다. 자동 크롤링이나 저장 코드를 새로 작성할 필요는 없습니다.

#### 요구사항

1. 제공 HTML에서 두 번째 책의 전체 제목, 가격 문자열, 제목 링크의 `href` 값을 찾아 답변란에 기록하세요.
2. `book_json_text`를 파싱하여 `book_data`에 저장하세요. 전체 자료형, 최상위 Key 목록, `books`의 자료형과 책 개수를 출력하세요.
3. 두 번째 책을 `second_book`에 저장하고 제목, `price` 안의 `text`, `detail_href`, 최상위 `source_url`을 출력하세요.
4. `second_book["price"]`와 `second_book["price"]["text"]`의 자료형을 출력하세요.
5. 두 번째 책의 제목·가격·상세 링크에 대해 HTML 값과 JSON 값을 나란히 정리하고, 같은 책의 정보인지 판단 근거를 적으세요.
6. 앞으로 만들 책 목록의 필드를 `title`, `price_text`, `detail_href`, `source_url`로 정한다고 가정하세요. 각 필드의 의미, 이번 단계의 Python 자료형, 값의 출처를 표로 설계하세요. 한 행이 무엇을 뜻하는지도 적으세요.
7. Q1~Q4에 답하세요.

> `price_text`는 앞으로 사용할 컬럼명입니다. 학습용 JSON에서는 `price` 안의 `text`에 저장되어 있으므로 원본의 위치와 최종 컬럼 이름이 다를 수 있습니다. 이 단계에서는 통화 기호 제거, URL 변환, DataFrame 생성 및 파일 저장을 수행하지 않습니다.

#### 해석 질문

**Q1.** 두 번째 책의 제목과 가격은 HTML과 학습용 JSON에서 각각 어디에 있나요?

**Q2.** 학습용 JSON의 `price`와 `price["text"]`는 각각 어떤 자료형인가요? 가격에 숫자가 보인다는 이유만으로 바로 숫자형이라고 판단해도 되나요?

**Q3.** 한 행을 책 한 권으로 구성할 때 `source_url`과 `detail_href`를 구분해서 기록하는 이유는 무엇인가요?

**Q4.** 필요한 책 정보를 제공하는 API가 있는 경우와, API 없이 HTML에만 정보가 있는 경우에는 각각 어떤 수집 방식을 검토할 수 있나요?

#### 제출 결과

- 두 번째 책의 HTML 관찰 기록
- JSON 구조·값 확인 코드와 출력 결과
- HTML·JSON 값 비교표와 판단 근거
- 필드 설계표와 한 행의 의미
- Q1~Q4 답변


### 작성 도움말

필수 2와 같은 방법으로 책 문자열을 파싱한 뒤 books 목록을 찾으세요. 두 번째 책을 선택한 다음 제목·가격·링크를 각각 읽습니다. 가격은 한 단계 더 들어 있고 출처는 최상위에 있으므로 모든 값을 같은 위치에서 찾지 않습니다.

필드 설계표의 ‘값의 출처’에는 단순히 HTML 또는 JSON이라고만 적기보다 **어떤 요소의 어떤 속성인지, 어떤 Key 안에 있는지**를 적으세요. 원본 가격·링크 문자열을 그대로 유지합니다.


### 제공 코드: 비교할 HTML과 JSON 확인


In [6]:
print("[HTML 예시]")
print(book_html)
print("\n[같은 책 정보를 재구성한 JSON]")
print(book_json_text)


[HTML 예시]

<article class="product_pod">
  <h3><a href="catalogue/a-light-in-the-attic_1000/index.html" title="A Light in the Attic">A Light in the ...</a></h3>
  <div class="product_price">
    <p class="price_color">£51.77</p>
  </div>
</article>

<article class="product_pod">
  <h3><a href="catalogue/tipping-the-velvet_999/index.html" title="Tipping the Velvet">Tipping the Velvet</a></h3>
  <div class="product_price">
    <p class="price_color">£53.74</p>
  </div>
</article>


[같은 책 정보를 재구성한 JSON]

{
  "source_url": "https://books.toscrape.com/",
  "books": [
    {
      "title": "A Light in the Attic",
      "price": {
        "text": "£51.77"
      },
      "detail_href": "catalogue/a-light-in-the-attic_1000/index.html"
    },
    {
      "title": "Tipping the Velvet",
      "price": {
        "text": "£53.74"
      },
      "detail_href": "catalogue/tipping-the-velvet_999/index.html"
    }
  ]
}



In [8]:
# 과제 1 코드를 작성하세요.
# 준비 셀의 book_json_text를 사용하세요.

# 1. 책 JSON 파싱 → 최상위 구조·책 목록 확인
# 2. `book_json_text`를 파싱하여 `book_data`에 저장하세요. 전체 자료형, 최상위 Key 목록, `books`의 자료형과 책 개수를 출력하세요.
book_data = json.loads(book_json_text)
print("전체 자료형:", type(book_data))
print("최상위 Key 목록:", list(book_data.keys()))
print("books 자료형:", type(book_data["books"]))
print("책 개수:", len(book_data["books"]))

# 2. 두 번째 책 선택 → 제목·중첩 가격·상세 링크 확인
# 3. 두 번째 책을 `second_book`에 저장하고 제목, `price` 안의 `text`, `detail_href`, 최상위 `source_url`을 출력하세요.
second_book = book_data["books"][1]
print("\n[두 번째 책]")
print("title:", second_book["title"])
print("price.text:", second_book["price"]["text"])
print("detail_href:", second_book["detail_href"])
print("source_url:", book_data["source_url"])   # 출처는 책 안이 아니라 최상위에 있음

# 3. 최상위 출처와 가격 관련 자료형 확인
# 4. `second_book["price"]`와 `second_book["price"]["text"]`의 자료형을 출력하세요.
print("\nprice 자료형:", type(second_book["price"]))
print("price.text 자료형:", type(second_book["price"]["text"]))

전체 자료형: <class 'dict'>
최상위 Key 목록: ['source_url', 'books']
books 자료형: <class 'list'>
책 개수: 2

[두 번째 책]
title: Tipping the Velvet
price.text: £53.74
detail_href: catalogue/tipping-the-velvet_999/index.html
source_url: https://books.toscrape.com/

price 자료형: <class 'dict'>
price.text 자료형: <class 'str'>


### 과제 1 관찰 및 답변 작성란

| 두 번째 책의 항목 | HTML에서 확인한 값 | JSON에서 확인한 값 |
|---|---|---|
| 전체 제목 | Tipping the Velvet (`a` 태그의 `title` 속성) | Tipping the Velvet (`books[1]["title"]`) |
| 가격 문자열 | £53.74 (`p.price_color`의 내용) | £53.74 (`books[1]["price"]["text"]`) |
| 상세 링크 | catalogue/tipping-the-velvet_999/index.html (`a` 태그의 `href` 속성) | catalogue/tipping-the-velvet_999/index.html (`books[1]["detail_href"]`) |

- 같은 책의 정보라고 판단한 근거: 전체 제목, 가격 문자열, 상세 링크 세 값이 HTML과 JSON에서 모두 동일하고, 둘 다 두 번째 책 영역(두 번째 `article`, `books[1]`)에 있음  
- 한 행이 나타내는 대상: 책 한 권 (HTML의 `article.product_pod` 하나, JSON의 `books` 안 항목 하나)  

| 설계할 필드 | 의미 | 이번 단계의 Python 자료형 | 값의 출처 |
|---|---|---|---|
| `title` | 책의 전체 제목 | str | HTML : `a` 태그의 `title` 속성 / JSON : `books[i]["title"]` |
| `price_text` | 화면에 표시된 가격 문자열 (통화 기호 포함) | str | HTML : `p.price_color`의 내용 / JSON : `books[i]["price"]["text"]` |
| `detail_href` | 책 상세 페이지로 가는 링크 경로 (상대 경로) | str | HTML : `a` 태그의 `href` 속성 / JSON : `books[i]["detail_href"]` |
| `source_url` | 이 책 정보를 가져온 목록 페이지 주소 | str | JSON : 최상위 `source_url` / HTML : 수집한 페이지 주소(`BOOKS_URL`) |

**Q1. 두 번째 책의 제목과 가격은 HTML과 학습용 JSON에서 각각 어디에 있나요?**

- 답변:  
    - HTML : 두 번째 article.product_pod 안에서, 제목은 h3 > a의 title 속성, 가격은 div.product_price > p.price_color의 내용에 있음  
    - 학습용 JSON : 최상위 books 목록의 두 번째 항목(books[1]) 안에서, 제목은 title Key, 가격은 price 안의 text Key에 있음  

**Q2. 학습용 JSON의 `price`와 `price["text"]`는 각각 어떤 자료형인가요? 가격에 숫자가 보인다는 이유만으로 바로 숫자형이라고 판단해도 되나요?**

- 답변:  
    - price : dict  
    - price["text"] : str  
    - 바로 숫자형이라고 판단하면 안 됨  
        -> £53.74는 숫자처럼 보여도 기호 £가 붙은 문자열이기 때문에 이대로는 숫자형처럼 더하기나 평균 같은 계산을 할 수 없음  

**Q3. 한 행을 책 한 권으로 구성할 때 `source_url`과 `detail_href`를 구분해서 기록하는 이유는 무엇인가요?**

- 답변: 두 값이 가리키는 대상과 역할이 다르기 때문  
    - source_url : 이 정보를 가져온 목록 페이지 주소 (https://books.toscrape.com/) -> 여러 책이 같은 값을 가짐  
    - detail_href : 그 책 한 권의 상세 페이지 경로 (catalogue/tipping-the-velvet_999/index.html) -> 책마다 다름  

**Q4. 필요한 책 정보를 제공하는 API가 있는 경우와, API 없이 HTML에만 정보가 있는 경우에는 각각 어떤 수집 방식을 검토할 수 있나요?**

- 답변:  
    - API가 있는 경우 : API에 요청해 받은 JSON을 json.loads()로 파싱하고, Key와 인덱스로 값을 꺼냄  
    - HTML에만 있는 경우 : HTML을 BeautifulSoup으로 파싱한 다음 책 영역(article.product_pod)을 찾고, 그 안의 태그·속성에서 값을 꺼내는 크롤링을 함

## 개념과 결과를 함께 정리하기

| 구분할 것 | 이번 실습에서 기억할 기준 |
|---|---|
| HTML 태그 / 속성 / 내용 | a는 태그, title·href는 속성, 태그 사이에는 표시 텍스트가 있음 |
| 직접 부모 / 더 바깥 조상 | 바로 한 단계 밖인지, 여러 단계 밖인지 구분 |
| JSON 문자열 / 파싱한 값 | 문자열을 json.loads()로 해석한 뒤 list·dict의 구조에 맞게 접근 |
| 전체 목록 / 항목 하나 / 항목 속 값 | posts는 목록, posts[0]은 한 게시글, 그 title은 제목 문자열 |
| 위치 번호 / 식별 번호 | [1]은 두 번째 위치, id는 게시글을 식별하는 필드의 값 |
| Key 이름 / Value | title이라는 이름과 실제 제목 글자는 다름 |
| 자료형 / 항목 수 | type()과 len()은 서로 다른 질문에 답함 |
| 값의 묶음 / 내부 문자열 | price는 dict, price["text"]는 str |
| 목록 출처 / 상세 링크 | 정보를 본 페이지 주소와 책 상세 페이지 경로를 구분 |
| 발췌·재구성 / 실제 전체 자료 | 예시 2개를 사이트 전체 개수로 해석하지 않음 |

### 막혔을 때 확인할 순서

1. 준비 셀을 실행하여 필요한 변수가 만들어졌는지 확인합니다.
2. JSON을 문자열 상태로 두고 Key에 접근하지 않았는지 확인합니다.
3. 현재 값이 list인지 dict인지 확인한 뒤 위치 번호와 Key 중 무엇을 사용할지 정합니다.
4. Key가 없다면 철자·대소문자와 현재 중첩 위치를 확인합니다. 없는 정보를 임의로 만들어 답하지 않습니다.
5. print() 결과를 읽을 때 종류·개수·실제 값을 구분합니다. 값만 보고 수집이나 저장까지 완료했다고 판단하지 않습니다.

이 강의의 목표는 함수를 많이 외우는 것이 아니라 **현재 어느 구조에 있고, 다음에 어떤 이름이나 위치를 선택해야 필요한 값에 도달하는지 설명하는 것**입니다.


---

## 실습 마무리

아래 질문에 자신의 말로 답하세요.

**Q1. 이번 실습에서 확인하려던 데이터와 구조상의 문제는 무엇이었나요?**

- 답변: 책의 제목, 가격, 상세 링크와 게시글의 번호, 작성자 번호, 제목, 본문이 어디있는지 확인했다.  
- HTML : 여러 태그가 섞여있음. JSON : 목록과 객체가 중첩되어있다.  
-> 한 항목에 대해서 경계와 값의 위치를 먼저 구분해야한다.

**Q2. HTML에서 데이터의 위치를 확인할 때 어떤 기준을 사용했나요?**

- 답변: 태그, class, title, href 등의 속성, 요소의 내용과 부모, 자식 관계를 확인함  
-> 책 한권을 보기 위해서 article.product_pod 영역을 먼저 찾고 제목과 가격 그 안에 요소를 찾았다.  

**Q3. JSON에서 Object와 Array는 Python의 어떤 자료형으로 바뀌며, 각각 어떻게 접근하나요?**

- 답변: object는 일반적으로 dict, Array는 list로 바뀐다.  
딕셔너리는 ['title']같은 key로 리스트는 [0],[1]같은 인덱스로 접근한다.

**Q4. HTML과 JSON에서 확인한 정보가 같은 책의 정보라고 판단한 근거는 무엇인가요?**

- 답변: 두번째 책 제목 Tippimg the Velvet, 가격 E53.74, 상세 링크가 두 표현에서 모두 일치  
-> HTML 정보를 재구성한 JSON과 대조하여 확인  

**Q5. 레코드, 필드, 데이터 구조 정의(스키마)는 이번 책 예시에서 각각 무엇을 의미하나요?**

- 답변: 레코드 : 책 한 권의 정보 묶음 필드 : 제목, 가격, 링크같은 개별 속성  
-> 여기서 스키마는 title, price_text, detail_href, source_url처럼 사용할 필드 이름과 의미 자료형 등을 정한 구조

**Q6. 이번 강의에서 확인한 내용은 다음 강의의 데이터 추출·저장에 어떻게 연결되나요?**

- 답변: 이번에는 사람이 구조를 이릭고 필요한 값의 위치를 확인했다.  
-> 다음 강의에서는 이 위치를 기준으로 값을 추출하여 리스트, 딕셔너리로 정리하고 datafram과 csv로 연결할 수 있다.


### 제출 전 확인

- 필수 1·2와 과제 1의 요구사항을 모두 수행했는지 확인하세요.
- 각 해석 질문과 실습 마무리 질문에 답했는지 확인하세요.
- 발췌 자료와 실제 전체 자료, 학습용 재구성과 원본 API를 구분했는지 확인하세요.
- 작성한 코드와 실행 결과가 남아 있는 `.ipynb` 파일을 제출하세요.
